In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install gdown


In [3]:
!gdown 1VGm-hWxktQp978mc6rEbkn9eDxo7zBRZ

# https://drive.google.com/file/d/1VGm-hWxktQp978mc6rEbkn9eDxo7zBRZ/view?usp=drive_link

Downloading...
From (original): https://drive.google.com/uc?id=1VGm-hWxktQp978mc6rEbkn9eDxo7zBRZ
From (redirected): https://drive.google.com/uc?id=1VGm-hWxktQp978mc6rEbkn9eDxo7zBRZ&confirm=t&uuid=401ae515-9302-4e73-86b7-b6eeb5778cb8
To: /content/suitesparse_kaggle_export.zip
100% 1.72G/1.72G [00:31<00:00, 55.5MB/s]


In [4]:
!unzip suitesparse_kaggle_export.zip


Streaming output truncated to the last 5000 lines.
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_1642.txt  
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_1643.txt  
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_1644.txt  
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_1645.txt  
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_1646.txt  
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_1647.txt  
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_1648.txt  
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_1649.txt  
  inflating: suitesparse_mtx/Moqri/MISKnowledgeMap/MISKnowledgeMap/MISKnowledgeMap_Abstract_165.txt  
  inflating: suitespars

In [5]:
!pip install ssgetpy -q


In [6]:
import json
import ssgetpy
from pathlib import Path

data_dir = Path("suitesparse_mtx")

# Load the saved metadata
with open("suitesparse_selected.json") as f:
    meta = json.load(f)

print("Matrices listed in JSON:", len(meta))

# Rebuild selected list exactly as in kaggle
selected = []
for item in meta:
    g = item["group"]
    n = item["name"]
    try:
        # Fetch the exact matrix entry from SuiteSparse DB metadata
        m = ssgetpy.search(group=g, name=n, limit=1)[0]
        selected.append(m)
    except:
        print("Could not find:", g, n)

print("Reconstructed selected list:", len(selected))


Matrices listed in JSON: 1765
Reconstructed selected list: 1765


In [ ]:
from pathlib import Path
import scipy.io
import scipy.sparse as sp
import numpy as np

data_dir = Path("suitesparse_mtx")

def load_matrix_metadata(m):
    """
    Given an ssgetpy Matrix object m (with .group and .name),
    recursively find a .mtx file under suitesparse_mtx/group/name/,
    load it as SciPy CSR (float64), and return it.
    """
    base_dir = data_dir / m.group / m.name

    if not base_dir.exists():
        print(f"[WARN] Directory not found for {m.group}/{m.name}: {base_dir}")
        return None

    # Recursively search for any .mtx file
    mtx_file = None
    for f in base_dir.rglob("*.mtx"):
        mtx_file = f
        break  # first match

    if mtx_file is None:
        print(f"[WARN] No .mtx file found under {base_dir}")
        return None

    print(f"Loading {m.group}/{m.name} from {mtx_file.relative_to(data_dir)}")
    A = scipy.io.mmread(str(mtx_file))
    A = A.tocsr().astype(np.float64)   # ensure CSR float64

    print(f"  shape: {A.shape}, nnz: {A.nnz}, dtype: {A.dtype}")
    return A

# Test on first 3 matrices
print("Testing loader on a few matrices...")
for m in selected[:3]:
    A = load_matrix_metadata(m)
    print("-" * 60)


Testing loader on a few matrices...
Loading HB/1138_bus from HB/1138_bus/1138_bus/1138_bus.mtx
  shape: (1138, 1138), nnz: 4054, dtype: float64
------------------------------------------------------------
Loading HB/494_bus from HB/494_bus/494_bus/494_bus.mtx
  shape: (494, 494), nnz: 1666, dtype: float64
------------------------------------------------------------
Loading HB/662_bus from HB/662_bus/662_bus/662_bus.mtx
  shape: (662, 662), nnz: 2474, dtype: float64
------------------------------------------------------------


In [8]:
# Install CuPy for CUDA 12 (works on current Colab GPUs)
!pip install -q cupy-cuda12x

import cupy as cp
import time
import scipy.sparse as sp
import numpy as np

def scipy_to_cupy_csr(A, dtype=cp.float64):
    """
    Convert a SciPy CSR matrix A to a CuPy CSR matrix with given dtype.
    """
    A = A.tocsr()
    data   = cp.asarray(A.data,   dtype=dtype)
    indices = cp.asarray(A.indices, dtype=cp.int32)
    indptr  = cp.asarray(A.indptr,  dtype=cp.int32)
    return cp.sparse.csr_matrix((data, indices, indptr), shape=A.shape)

def gpu_reference_spmv(A, n_runs=5):
    """
    Reference SpMV on GPU in float64 precision.

    Returns:
        A64_gpu : CuPy CSR (float64)
        x_ref   : CuPy dense vector (float64)
        y_ref   : CuPy dense vector (float64)
        avg_t   : average execution time over n_runs (seconds)
    """
    # 1) Move A to GPU in float64
    A64_gpu = scipy_to_cupy_csr(A, dtype=cp.float64)

    n = A.shape[1]
    x_ref = cp.random.randn(n, dtype=cp.float64)

    # 2) Warm-up (to avoid including one-time overhead)
    _ = A64_gpu @ x_ref
    cp.cuda.Stream.null.synchronize()

    # 3) Timed runs
    t0 = time.time()
    for _ in range(n_runs):
        y_ref = A64_gpu @ x_ref
    cp.cuda.Stream.null.synchronize()

    avg_t = (time.time() - t0) / n_runs
    return A64_gpu, x_ref, y_ref, avg_t

# Quick sanity test on the first matrix
m_test = selected[0]
A_test = load_matrix_metadata(m_test)

A64_gpu, x_ref, y_ref, t_ref = gpu_reference_spmv(A_test)
print(f"Test matrix: {m_test.group}/{m_test.name}")
print("  shape:", A_test.shape, "nnz:", A_test.nnz)
print(f"  Reference GPU SpMV avg time: {t_ref:.6e} s")
print("  ||y_ref|| (L2 norm):", float(cp.linalg.norm(y_ref)))


Loading HB/1138_bus from HB/1138_bus/1138_bus/1138_bus.mtx
  shape: (1138, 1138), nnz: 4054, dtype: float64
Test matrix: HB/1138_bus
  shape: (1138, 1138) nnz: 4054
  Reference GPU SpMV avg time: 1.334667e-04 s
  ||y_ref|| (L2 norm): 130134.42525594219


In [ ]:
import scipy.sparse as sp

def build_entrywise_matrices(A, thresh=1.0):
    """
    Split entries of A by magnitude:

      - A_hi: |a_ij| >= thresh
      - A_lo: |a_ij| <  thresh

    Returns two SciPy CSR matrices.
    """
    A = A.tocoo()
    data = A.data
    rows = A.row
    cols = A.col

    mask_hi = np.abs(data) >= thresh
    mask_lo = ~mask_hi

    A_hi = sp.coo_matrix((data[mask_hi], (rows[mask_hi], cols[mask_hi])),
                         shape=A.shape).tocsr()
    A_lo = sp.coo_matrix((data[mask_lo], (rows[mask_lo], cols[mask_lo])),
                         shape=A.shape).tocsr()
    return A_hi, A_lo

def eval_entrywise_spmv(A, x_ref, y_ref, thresh=1.0, n_runs=5):
    """
    Entry-wise mixed precision SpMV:

      - A_hi stored and multiplied in float64
      - A_lo stored and multiplied in float32
      - Both results accumulated in float64

    A: SciPy sparse matrix (CSR/COO/etc)
    x_ref, y_ref: CuPy vectors from gpu_reference_spmv
    """
    # 1) Split into high/low magnitude matrices on CPU
    A_hi, A_lo = build_entrywise_matrices(A, thresh=thresh)

    # 2) Move to GPU with desired precisions
    A_hi_gpu = scipy_to_cupy_csr(A_hi, dtype=cp.float64)
    A_lo_gpu = scipy_to_cupy_csr(A_lo, dtype=cp.float32)

    # 3) Prepare x in both precisions
    x64 = x_ref
    x32 = x_ref.astype(cp.float32)

    # 4) Warm-up
    _ = A_hi_gpu @ x64
    _ = A_lo_gpu @ x32
    cp.cuda.Stream.null.synchronize()

    # 5) Timed runs
    t0 = time.time()
    for _ in range(n_runs):
        y_hi = A_hi_gpu @ x64
        y_lo = A_lo_gpu @ x32
        y    = y_hi + y_lo.astype(cp.float64)
    cp.cuda.Stream.null.synchronize()
    avg_t = (time.time() - t0) / n_runs

    # 6) Relative error vs reference
    rel_err = cp.linalg.norm(y - y_ref) / cp.linalg.norm(y_ref)
    return avg_t, float(rel_err)

#  Quick test on the same test matrix as before
m_test = selected[0]
A_test = load_matrix_metadata(m_test)

A64_gpu, x_ref, y_ref, t_ref = gpu_reference_spmv(A_test)
print(f"Reference time (float64): {t_ref:.6e} s")

t_entry, e_entry = eval_entrywise_spmv(A_test, x_ref, y_ref, thresh=1.0)
print(f"Entry-wise time: {t_entry:.6e} s,  rel_err = {e_entry:.3e}")


Loading HB/1138_bus from HB/1138_bus/1138_bus/1138_bus.mtx
  shape: (1138, 1138), nnz: 4054, dtype: float64
Reference time (float64): 1.034260e-04 s
Entry-wise time: 4.934797e-02 s,  rel_err = 2.964e-12


In [ ]:
def build_rowwise_matrices(A, frac_high=0.3):
    """
    Split rows by importance using max |a_ij| per row.

    Top frac_high fraction of rows go to high precision (A_hi),
    remaining rows go to low precision (A_lo).
    """
    A = A.tocsr()
    n_rows = A.shape[0]
    row_max = np.zeros(n_rows, dtype=np.float64)

    data = A.data
    indptr = A.indptr

    # Compute max |a_ij| per row
    for i in range(n_rows):
        s, e = indptr[i], indptr[i+1]
        if s == e:
            row_max[i] = 0.0
        else:
            row_max[i] = np.max(np.abs(data[s:e]))

    # Determine threshold to pick top frac_high rows
    q = np.quantile(row_max, 1.0 - frac_high)
    hi_mask = row_max >= q
    lo_mask = ~hi_mask

    A_hi = A[hi_mask, :]
    A_lo = A[lo_mask, :]

    return A_hi, A_lo, hi_mask, lo_mask

def eval_rowwise_spmv(A, x_ref, y_ref, frac_high=0.3, n_runs=5):
    """
    Row-wise mixed precision SpMV:

      - High-importance rows in float64 (A_hi)
      - Remaining rows in float32 (A_lo)

    A: SciPy sparse matrix
    x_ref, y_ref: CuPy vectors from gpu_reference_spmv
    """
    # 1) Build high/low row blocks
    A_hi, A_lo, hi_mask, lo_mask = build_rowwise_matrices(A, frac_high=frac_high)

    # 2) Move row blocks to GPU
    A_hi_gpu = scipy_to_cupy_csr(A_hi, dtype=cp.float64)
    A_lo_gpu = scipy_to_cupy_csr(A_lo, dtype=cp.float32)

    x64 = x_ref
    x32 = x_ref.astype(cp.float32)

    # 3) Warm-up
    _ = A_hi_gpu @ x64
    _ = A_lo_gpu @ x32
    cp.cuda.Stream.null.synchronize()

    # 4) Timed runs
    t0 = time.time()
    for _ in range(n_runs):
        y_hi = A_hi_gpu @ x64   # high-precision rows
        y_lo = A_lo_gpu @ x32   # low-precision rows
    cp.cuda.Stream.null.synchronize()
    avg_t = (time.time() - t0) / n_runs

    # 5) Reconstruct full y in original row order
    y_full = cp.zeros_like(y_ref)
    hi_idx = np.where(hi_mask)[0]
    lo_idx = np.where(lo_mask)[0]

    y_full[hi_idx] = y_hi
    y_full[lo_idx] = y_lo.astype(cp.float64)

    # 6) Relative error vs reference
    rel_err = cp.linalg.norm(y_full - y_ref) / cp.linalg.norm(y_ref)
    return avg_t, float(rel_err)

#  Quick test on the same test matrix
m_test = selected[0]
A_test = load_matrix_metadata(m_test)

A64_gpu, x_ref, y_ref, t_ref = gpu_reference_spmv(A_test)
print(f"Reference time (float64): {t_ref:.6e} s")

t_row, e_row = eval_rowwise_spmv(A_test, x_ref, y_ref, frac_high=0.3)
print(f"Row-wise time: {t_row:.6e} s,  rel_err = {e_row:.3e}")


Loading HB/1138_bus from HB/1138_bus/1138_bus/1138_bus.mtx
  shape: (1138, 1138), nnz: 4054, dtype: float64
Reference time (float64): 1.039505e-04 s
Row-wise time: 1.458645e-04 s,  rel_err = 1.150e-09


In [11]:
def build_adaptive_matrices(A, quantile=0.7):
    """
    2-bucket adaptive split by |a_ij|:
      - A_hi: entries >= quantile(|a|)
      - A_lo: entries <  quantile(|a|)
    """
    A = A.tocoo()
    data = A.data
    rows = A.row
    cols = A.col

    abs_data = np.abs(data)
    if abs_data.size == 0:
        Z = sp.csr_matrix(A.shape, dtype=np.float64)
        return Z, Z

    thresh = np.quantile(abs_data, quantile)

    mask_hi = abs_data >= thresh
    mask_lo = abs_data <  thresh

    A_hi = sp.coo_matrix((data[mask_hi], (rows[mask_hi], cols[mask_hi])),
                         shape=A.shape).tocsr()
    A_lo = sp.coo_matrix((data[mask_lo], (rows[mask_lo], cols[mask_lo])),
                         shape=A.shape).tocsr()

    return A_hi, A_lo

def eval_adaptive_spmv(A, x_ref, y_ref, quantile=0.7, n_runs=5):
    """
    Adaptive mixed precision SpMV:

      - High bucket (>= quantile) -> float64
      - Low bucket  (< quantile)  -> float32
    """
    # 1) Build high/low matrices
    A_hi, A_lo = build_adaptive_matrices(A, quantile=quantile)

    # 2) Move to GPU with desired precision
    A_hi_gpu = scipy_to_cupy_csr(A_hi, dtype=cp.float64)
    A_lo_gpu = scipy_to_cupy_csr(A_lo, dtype=cp.float32)

    x64 = x_ref
    x32 = x_ref.astype(cp.float32)

    # 3) Warm-up
    _ = A_hi_gpu @ x64
    _ = A_lo_gpu @ x32
    cp.cuda.Stream.null.synchronize()

    # 4) Timed runs
    t0 = time.time()
    for _ in range(n_runs):
        y_hi = A_hi_gpu @ x64
        y_lo = A_lo_gpu @ x32
        y    = y_hi + y_lo.astype(cp.float64)
    cp.cuda.Stream.null.synchronize()

    avg_t  = (time.time() - t0) / n_runs
    rel_err = cp.linalg.norm(y - y_ref) / cp.linalg.norm(y_ref)
    return avg_t, float(rel_err)

# Quick test on the same test matrix
m_test = selected[0]
A_test = load_matrix_metadata(m_test)

A64_gpu, x_ref, y_ref, t_ref = gpu_reference_spmv(A_test)
print(f"Reference time (float64): {t_ref:.6e} s")

t_adapt, e_adapt = eval_adaptive_spmv(A_test, x_ref, y_ref, quantile=0.7)
print(f"Adaptive time: {t_adapt:.6e} s,  rel_err = {e_adapt:.3e}")


Loading HB/1138_bus from HB/1138_bus/1138_bus/1138_bus.mtx
  shape: (1138, 1138), nnz: 4054, dtype: float64
Reference time (float64): 8.831024e-05 s
Adaptive time: 2.151489e-04 s,  rel_err = 8.023e-10


In [12]:
# ===== Simple, robust labeling loop using your existing loader =====

import json
from tqdm.auto import tqdm

err_tol = 1e-3   # maximum allowed relative error
labels = {}      # "group/name" -> 0/1/2
stats  = {}      # detailed timings + errors

skipped_error = []   # matrices that failed for any reason

for m in tqdm(selected, desc="Labeling matrices"):
    name = f"{m.group}/{m.name}"

    try:
        # 1) Load matrix using your existing helper
        A = load_matrix_metadata(m)
        if A is None:
            print(f"[SKIP] Could not load {name}")
            skipped_error.append(name)
            continue

        # 2) Reference SpMV (float64 on GPU)
        A64_gpu, x_ref, y_ref, t_ref = gpu_reference_spmv(A)

        # 3) Evaluate three strategies
        t_ent, e_ent = eval_entrywise_spmv(A, x_ref, y_ref, thresh=1.0)
        t_row, e_row = eval_rowwise_spmv(A, x_ref, y_ref, frac_high=0.3)
        t_adp, e_adp = eval_adaptive_spmv(A, x_ref, y_ref, quantile=0.7)

        methods = [
            {"name": "entrywise", "time": t_ent, "err": e_ent, "label": 0},
            {"name": "rowwise",   "time": t_row, "err": e_row, "label": 1},
            {"name": "adaptive",  "time": t_adp, "err": e_adp, "label": 2},
        ]

        # 4) Save raw stats
        stats[name] = methods

        # 5) Filter by error tolerance
        feasible = [mt for mt in methods if mt["err"] <= err_tol]

        if feasible:
            # Among feasible ones, pick the fastest
            best = min(feasible, key=lambda d: d["time"])
        else:
            # If nothing meets error tolerance, pick the one with smallest error
            best = min(methods, key=lambda d: d["err"])

        # 6) Store best label
        labels[name] = best["label"]

    except Exception as e:
        # This will catch things like *_label.mtx becoming numpy.ndarray inside load_matrix_metadata
        print(f"[ERROR] {name}: {e}")
        skipped_error.append(name)
        continue

# 7) Save results
with open("labels.json", "w") as f:
    json.dump(labels, f, indent=2)

with open("stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print("Done. Labeled matrices:", len(labels))
print("Label distribution:", {
    lbl: list(labels.values()).count(lbl) for lbl in sorted(set(labels.values()))
})

print("\nSkipped due to errors (including label/metadata matrices):", len(skipped_error))


Labeling matrices:   0%|          | 0/1765 [00:00<?, ?it/s]

Loading HB/1138_bus from HB/1138_bus/1138_bus/1138_bus.mtx
  shape: (1138, 1138), nnz: 4054, dtype: float64
Loading HB/494_bus from HB/494_bus/494_bus/494_bus.mtx
  shape: (494, 494), nnz: 1666, dtype: float64
Loading HB/662_bus from HB/662_bus/662_bus/662_bus.mtx
  shape: (662, 662), nnz: 2474, dtype: float64
Loading HB/685_bus from HB/685_bus/685_bus/685_bus.mtx
  shape: (685, 685), nnz: 3249, dtype: float64
Loading HB/arc130 from HB/arc130/arc130/arc130.mtx
  shape: (130, 130), nnz: 1282, dtype: float64
Loading HB/bcsstk01 from HB/bcsstk01/bcsstk01/bcsstk01.mtx
  shape: (48, 48), nnz: 400, dtype: float64
Loading HB/bcsstk02 from HB/bcsstk02/bcsstk02/bcsstk02.mtx
  shape: (66, 66), nnz: 4356, dtype: float64
Loading HB/bcsstk03 from HB/bcsstk03/bcsstk03/bcsstk03.mtx
  shape: (112, 112), nnz: 640, dtype: float64
Loading HB/bcsstk04 from HB/bcsstk04/bcsstk04/bcsstk04.mtx
  shape: (132, 132), nnz: 3648, dtype: float64
Loading HB/bcsstk05 from HB/bcsstk05/bcsstk05/bcsstk05.mtx
  shape: (1

In [13]:
import numpy as np
import json
from tqdm import tqdm

def sparse_to_image(A, H=128, W=128):
    """
    Convert a sparse matrix A into a fixed-size HxW image
    using coordinate binning + normalized magnitude.
    """
    A = A.tocoo()
    n_rows, n_cols = A.shape

    if A.nnz == 0:
        return np.zeros((H, W), dtype=np.float32)

    rows = A.row.astype(np.float64)
    cols = A.col.astype(np.float64)
    vals = A.data.astype(np.float64)

    # 1) Scale coordinates to [0, H-1] and [0, W-1]
    r_scaled = np.floor(rows * (H - 1) / max(n_rows - 1, 1)).astype(int)
    c_scaled = np.floor(cols * (W - 1) / max(n_cols - 1, 1)).astype(int)

    # 2) Normalize |vals| to [0,1]
    v_abs = np.abs(vals)
    v_min, v_max = v_abs.min(), v_abs.max()
    if v_max - v_min < 1e-12:
        v_norm = np.zeros_like(v_abs)
    else:
        v_norm = (v_abs - v_min) / (v_max - v_min)

    # 3) Accumulate into image
    img = np.zeros((H, W), dtype=np.float32)
    np.add.at(img, (r_scaled, c_scaled), v_norm)

    # 4) Normalize whole image to [0,1]
    max_val = img.max()
    if max_val > 0:
        img /= max_val

    return img

# --- Load labels ---
with open("labels.json", "r") as f:
    labels = json.load(f)

H = W = 128
X_list = []
y_list = []
name_list = []

print("Total labeled matrices:", len(labels))

for m in tqdm(selected, desc="Building images"):
    name = f"{m.group}/{m.name}"
    if name not in labels:
        continue

    A = load_matrix_metadata(m)
    if A is None:
        continue

    img = sparse_to_image(A, H=H, W=W)
    X_list.append(img)
    y_list.append(labels[name])
    name_list.append(name)

X = np.stack(X_list, axis=0).astype(np.float32)   # (N, 128, 128)
y = np.array(y_list, dtype=np.int64)              # (N,)

np.save("X_images.npy", X)
np.save("y_labels.npy", y)

print("Final dataset shape:")
print("  X:", X.shape)
print("  y:", y.shape)
print("Label counts:", {int(k): int((y == k).sum()) for k in np.unique(y)})


Total labeled matrices: 1211


Building images:   1%|          | 19/1765 [00:00<00:09, 189.88it/s]

Loading HB/1138_bus from HB/1138_bus/1138_bus/1138_bus.mtx
  shape: (1138, 1138), nnz: 4054, dtype: float64
Loading HB/494_bus from HB/494_bus/494_bus/494_bus.mtx
  shape: (494, 494), nnz: 1666, dtype: float64
Loading HB/662_bus from HB/662_bus/662_bus/662_bus.mtx
  shape: (662, 662), nnz: 2474, dtype: float64
Loading HB/685_bus from HB/685_bus/685_bus/685_bus.mtx
  shape: (685, 685), nnz: 3249, dtype: float64
Loading HB/arc130 from HB/arc130/arc130/arc130.mtx
  shape: (130, 130), nnz: 1282, dtype: float64
Loading HB/bcsstk01 from HB/bcsstk01/bcsstk01/bcsstk01.mtx
  shape: (48, 48), nnz: 400, dtype: float64
Loading HB/bcsstk02 from HB/bcsstk02/bcsstk02/bcsstk02.mtx
  shape: (66, 66), nnz: 4356, dtype: float64
Loading HB/bcsstk03 from HB/bcsstk03/bcsstk03/bcsstk03.mtx
  shape: (112, 112), nnz: 640, dtype: float64
Loading HB/bcsstk04 from HB/bcsstk04/bcsstk04/bcsstk04.mtx
  shape: (132, 132), nnz: 3648, dtype: float64
Loading HB/bcsstk05 from HB/bcsstk05/bcsstk05/bcsstk05.mtx
  shape: (1

Building images:   2%|▏         | 38/1765 [00:00<00:26, 64.17it/s] 

Loading HB/bcsstk26 from HB/bcsstk26/bcsstk26/bcsstk26.mtx
  shape: (1922, 1922), nnz: 30336, dtype: float64
Loading HB/bcsstk27 from HB/bcsstk27/bcsstk27/bcsstk27.mtx
  shape: (1224, 1224), nnz: 56126, dtype: float64
Loading HB/bcsstk28 from HB/bcsstk28/bcsstk28/bcsstk28.mtx
  shape: (4410, 4410), nnz: 219024, dtype: float64
Loading HB/bcsstm01 from HB/bcsstm01/bcsstm01/bcsstm01.mtx
  shape: (48, 48), nnz: 48, dtype: float64
Loading HB/bcsstm02 from HB/bcsstm02/bcsstm02/bcsstm02.mtx
  shape: (66, 66), nnz: 66, dtype: float64
Loading HB/bcsstm03 from HB/bcsstm03/bcsstm03/bcsstm03.mtx
  shape: (112, 112), nnz: 112, dtype: float64
Loading HB/bcsstm04 from HB/bcsstm04/bcsstm04/bcsstm04.mtx
  shape: (132, 132), nnz: 132, dtype: float64
Loading HB/bcsstm05 from HB/bcsstm05/bcsstm05/bcsstm05.mtx
  shape: (153, 153), nnz: 153, dtype: float64
Loading HB/bcsstm06 from HB/bcsstm06/bcsstm06/bcsstm06.mtx
  shape: (420, 420), nnz: 420, dtype: float64
Loading HB/bcsstm07 from HB/bcsstm07/bcsstm07/bc

...

Loading ML_Graph/k49_norm_10NN from ML_Graph/k49_norm_10NN/k49_norm_10NN/k49_norm_10NN.mtx
  shape: (38547, 38547), nnz: 618158, dtype: float64
Loading ML_Graph/mice_10NN from ML_Graph/mice_10NN/mice_10NN/mice_10NN.mtx
  shape: (1077, 1077), nnz: 13484, dtype: float64
Loading ML_Graph/micromass_10NN from ML_Graph/micromass_10NN/micromass_10NN/micromass_10NN.mtx
  shape: (571, 571), nnz: 9668, dtype: float64
Loading ML_Graph/Olivetti_norm_10NN from ML_Graph/Olivetti_norm_10NN/Olivetti_norm_10NN/Olivetti_norm_10NN.mtx
  shape: (400, 400), nnz: 5656, dtype: float64
Loading ML_Graph/Plants_10NN from ML_Graph/Plants_10NN/Plants_10NN/Plants_10NN.mtx
  shape: (1600, 1600), nnz: 21930, dtype: float64
Loading ML_Graph/Spectro_10NN from ML_Graph/Spectro_10NN/Spectro_10NN/Spectro_10NN.mtx
  shape: (531, 531), nnz: 7422, dtype: float64
Loading ML_Graph/umistfacesnorm_10NN from ML_Graph/umistfacesnorm_10NN/umistfacesnorm_10NN/umistfacesnorm_10NN.mtx
  shape: (575, 575), nnz: 6990, dtype: float64
Lo

Building images: 100%|█████████▉| 1762/1765 [00:34<00:00, 64.23it/s]

  shape: (100000, 100000), nnz: 797974, dtype: float64
Loading FlowIPM22/uni_chimera_i3 from FlowIPM22/uni_chimera_i3/uni_chimera_i3/uni_chimera_i3_A_08.mtx
  shape: (100000, 100000), nnz: 499696, dtype: float64
Loading FlowIPM22/uni_chimera_i4 from FlowIPM22/uni_chimera_i4/uni_chimera_i4/uni_chimera_i4_A_11.mtx
  shape: (100000, 100000), nnz: 814436, dtype: float64


Building images: 100%|██████████| 1765/1765 [00:35<00:00, 49.72it/s]

Loading FlowIPM22/uni_chimera_i5 from FlowIPM22/uni_chimera_i5/uni_chimera_i5/uni_chimera_i5_A_10.mtx
  shape: (100000, 100000), nnz: 499982, dtype: float64
Final dataset shape:
  X: (1241, 128, 128)
  y: (1241,)
Label counts: {0: 59, 1: 1157, 2: 25}


In [14]:
import numpy as np

# Load original (unbalanced) dataset
X = np.load("X_images.npy")      # shape: (N, 128, 128)
y = np.load("y_labels.npy")      # shape: (N,)

print("Original X shape:", X.shape)
print("Original y shape:", y.shape)

unique, counts = np.unique(y, return_counts=True)
print("Original class counts:")
for c, cnt in zip(unique, counts):
    print(f"  class {c}: {cnt}")


Original X shape: (1241, 128, 128)
Original y shape: (1241,)
Original class counts:
  class 0: 59
  class 1: 1157
  class 2: 25


In [15]:
import numpy as np

def augment_sparse_image(img, noise_std=0.03, drop_prob=0.01):
    """
    Structure-aware augmentation for sparse matrix images.
    - img: (H, W) in [0,1]
    - Add small noise only on non-zero pixels
    - Randomly drop a tiny fraction of non-zero pixels
    """
    aug = img.copy()

    # Non-zero structure mask
    mask = aug > 0

    if mask.sum() == 0:
        # If matrix is completely empty, do nothing
        return aug

    # 1) Small Gaussian noise on non-zero pixels
    noise = np.random.normal(loc=0.0, scale=noise_std, size=aug.shape).astype(np.float32)
    aug[mask] = aug[mask] + noise[mask]

    # 2) Randomly drop a small fraction of non-zero pixels
    drop_mask = (np.random.rand(*aug.shape) < drop_prob) & mask
    aug[drop_mask] = 0.0

    # 3) Clip back to [0,1]
    aug = np.clip(aug, 0.0, 1.0)

    return aug


In [16]:
N_AUG = 5  # number of augmentations per minority sample

X_list = []
y_list = []

# 1) Add all original samples
for i in range(len(X)):
    X_list.append(X[i])
    y_list.append(int(y[i]))

# 2) Augment minority classes (0 and 2)
minority_classes = [0, 2]

for cls in minority_classes:
    idxs = np.where(y == cls)[0]
    print(f"Class {cls}: original count = {len(idxs)}")

    for idx in idxs:
        img = X[idx]
        for _ in range(N_AUG):
            aug_img = augment_sparse_image(img)
            X_list.append(aug_img)
            y_list.append(cls)

X_aug = np.stack(X_list, axis=0).astype(np.float32)
y_aug = np.array(y_list, dtype=np.int64)

print("Augmented X shape:", X_aug.shape)
print("Augmented y shape:", y_aug.shape)

unique, counts = np.unique(y_aug, return_counts=True)
print("Augmented class counts:")
for c, cnt in zip(unique, counts):
    print(f"  class {c}: {cnt}")


Class 0: original count = 59
Class 2: original count = 25
Augmented X shape: (1661, 128, 128)
Augmented y shape: (1661,)
Augmented class counts:
  class 0: 354
  class 1: 1157
  class 2: 150


In [17]:
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split

print("Final augmented dataset shape:", X_aug.shape, y_aug.shape)

# Convert to tensors; add channel dim (N, 1, 128, 128)
X_t = torch.from_numpy(X_aug).unsqueeze(1).float()   # (N, 1, 128, 128)
y_t = torch.from_numpy(y_aug).long()                 # (N,)

dataset = TensorDataset(X_t, y_t)

# 80/20 train/val split
N = len(dataset)
n_train = int(0.8 * N)
n_val = N - n_train

train_ds, val_ds = random_split(dataset, [n_train, n_val])

print("Total samples:", N)
print("Train size:", len(train_ds), "Val size:", len(val_ds))

batch_size = 32

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)


Final augmented dataset shape: (1661, 128, 128) (1661,)
Total samples: 1661
Train size: 1328 Val size: 333


In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PrecisionCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: Capture fine details
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),   # Output: (32, 64, 64)

            # Block 2: Intermediate features
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),   # Output: (64, 32, 32)

            # Block 3: Deep semantic features (The New Layer)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)    # Output: (128, 16, 16)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            # Corrected dimensions: 128 filters * 16 * 16 spatial size
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.5), # Dropout is good for regularization
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PrecisionCNN(num_classes=3).to(device)

print(f"Model created on {device}. Flatten size: {128*16*16}")


Model created on cuda. Flatten size: 32768


In [19]:
!nvidia --smi


/bin/bash: line 1: nvidia: command not found


In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        """
        alpha: Class weights (tensor). pass your 'class_weights_tensor' here.
        gamma: Focusing parameter. Higher gamma (e.g., 2.0) focuses more on hard examples.
        """
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # 1. Calculate standard Cross Entropy Loss
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)

        # 2. Get the probabilities for the correct class (p_t)
        pt = torch.exp(-ce_loss)

        # 3. Calculate Focal Loss: (1 - pt)^gamma * CE_Loss
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [21]:


import torch
import numpy as np
# y_aug is still in NumPy form from earlier
classes = np.array([0, 1, 2], dtype=np.int64)
class_counts = np.array([(y_aug == c).sum() for c in classes], dtype=np.float32)

print("Class counts (augmented):", dict(zip(classes, class_counts)))

# Inverse frequency weights
# class_weights = 1.0 / class_counts
class_weights = 1.0 / np.sqrt(class_counts)

# Optional: normalize so that average weight ≈ 1
class_weights = class_weights / class_weights.sum() * len(classes)

print("Class weights:", dict(zip(classes, class_weights)))

class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(device)

# criterion = nn.CrossEntropyLoss(weight=class_weights_t)
criterion = FocalLoss(alpha=class_weights_t, gamma=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# LR scheduler: Reduce LR when val_loss plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,      # reduce LR by half
    patience=5,      # epochs with no improvement before reducing LR
    min_lr=1e-6,     # don't go below this

)



Class counts (augmented): {np.int64(0): np.float32(354.0), np.int64(1): np.float32(1157.0), np.int64(2): np.float32(150.0)}
Class weights: {np.int64(0): np.float32(0.97107214), np.int64(1): np.float32(0.5371387), np.int64(2): np.float32(1.4917892)}


In [22]:
from tqdm import tqdm
import copy

def run_epoch(loader, model, optimizer=None, device=device):
    if optimizer is None:
        model.eval()
    else:
        model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == yb).sum().item()
        total_samples += xb.size(0)

    avg_loss = total_loss / total_samples
    acc = total_correct / total_samples
    return avg_loss, acc


EPOCHS = 80

#Early stopping setup
patience = 10              # how many epochs to wait after last best val_loss
best_val_loss = float('inf')
best_state_dict = copy.deepcopy(model.state_dict())
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, model, optimizer)
    val_loss, val_acc     = run_epoch(val_loader, model, optimizer=None)

    #  Step the LR scheduler based on validation loss
    scheduler.step(val_loss)

    # Get current learning rate (for logging)
    current_lr = optimizer.param_groups[0]['lr']

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.3f} | "
        f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.3f} | "
        f"LR: {current_lr:.6f}"
    )

    #  Early stopping logic
    if val_loss < best_val_loss - 1e-4:  # small delta to avoid noise
        best_val_loss = val_loss
        best_state_dict = copy.deepcopy(model.state_dict())
        patience_counter = 0
        # print("  ↳ New best model found, saving state dict.")
    else:
        patience_counter += 1
        # print(f"  ↳ No improvement. Patience: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

# Restore best model weights
model.load_state_dict(best_state_dict)

# (Optional) Save best model to disk
torch.save(model.state_dict(), "best_precision_cnn.pth")
print("Best model weights restored and saved to best_precision_cnn.pth")


Epoch 01 | Train Loss: 0.5172, Acc: 0.544 | Val Loss: 0.3988, Acc: 0.637 | LR: 0.000100
Epoch 02 | Train Loss: 0.3436, Acc: 0.651 | Val Loss: 0.3116, Acc: 0.793 | LR: 0.000100
Epoch 03 | Train Loss: 0.3006, Acc: 0.691 | Val Loss: 0.2643, Acc: 0.661 | LR: 0.000100
Epoch 04 | Train Loss: 0.2718, Acc: 0.703 | Val Loss: 0.2397, Acc: 0.760 | LR: 0.000100
Epoch 05 | Train Loss: 0.2518, Acc: 0.756 | Val Loss: 0.2168, Acc: 0.769 | LR: 0.000100
Epoch 06 | Train Loss: 0.2196, Acc: 0.755 | Val Loss: 0.2062, Acc: 0.859 | LR: 0.000100
Epoch 07 | Train Loss: 0.2078, Acc: 0.762 | Val Loss: 0.1839, Acc: 0.880 | LR: 0.000100
Epoch 08 | Train Loss: 0.1907, Acc: 0.794 | Val Loss: 0.1897, Acc: 0.745 | LR: 0.000100
Epoch 09 | Train Loss: 0.1883, Acc: 0.811 | Val Loss: 0.1458, Acc: 0.859 | LR: 0.000100
Epoch 10 | Train Loss: 0.1616, Acc: 0.837 | Val Loss: 0.1323, Acc: 0.889 | LR: 0.000100
Epoch 11 | Train Loss: 0.1384, Acc: 0.856 | Val Loss: 0.1288, Acc: 0.880 | LR: 0.000100
Epoch 12 | Train Loss: 0.1460, A

In [23]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

model.eval()
all_preds = []
all_trues = []

with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        preds = logits.argmax(dim=1)

        all_preds.append(preds.cpu().numpy())
        all_trues.append(yb.cpu().numpy())

y_true = np.concatenate(all_trues)
y_pred = np.concatenate(all_preds)

print("True labels: ", y_true)
print("Pred labels:", y_pred)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
print("\nConfusion Matrix (rows = true, cols = predicted):")
print(cm)

target_names = ["Entry-wise (0)", "Row-wise (1)", "Adaptive (2)"]
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=target_names, digits=3))


True labels:  [1 1 2 1 1 1 1 1 1 1 1 0 1 0 0 1 2 1 1 1 1 1 1 1 0 1 1 2 0 1 1 0 1 1 1 1 0
 0 1 0 1 1 0 1 1 0 1 1 1 0 1 1 1 0 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 0 0 1 1 1
 1 0 1 1 2 1 1 1 1 1 1 2 2 1 1 1 1 1 1 1 1 1 0 1 1 2 1 2 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 0 1 0 2 0 1 1 1 1 0 1 1 0 1 1 1 1 1 0 1 1 1 1 1 1 1 1 0 2 1 1
 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 0 1 0 1 1 0 0 1 1 1 0 2 1 0 1 1 1 1 1 0 1 0
 1 2 1 1 1 0 1 1 1 1 2 0 1 0 0 1 1 2 0 1 1 1 1 0 0 1 1 0 1 1 1 1 2 1 0 1 1
 0 2 2 1 1 2 1 0 0 1 0 1 2 1 1 1 1 0 1 1 1 1 0 0 2 1 2 2 1 1 0 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 0 2 0 1 1 1 1 2 1 1 1 1 1 0 1 1 1 1 1 2 0 2 1 1 0 1 1 1 1
 1 1 1 1 0 0 0 0 0 1 0 1 2 0 1 1 1 1 1 0 1 1 1 1 1 1 0 0 2 1 1 1 1 1 2 1 0]
Pred labels: [1 1 2 1 1 1 1 1 1 1 1 0 1 0 0 1 2 1 1 1 1 1 1 1 0 1 1 2 1 1 1 0 1 1 1 1 0
 0 1 0 1 1 0 1 1 0 0 1 1 0 1 1 1 0 1 1 1 0 1 1 1 1 0 0 1 1 1 1 1 0 0 1 1 1
 0 0 1 1 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 2 1 2 1 1 1 1 1 1 1 1 0
 0 1 1 1 1 1 0 1 1 2 0 1 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 1

In [24]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Check if you already have 'val_loader' from training. If so, use it!
if 'val_loader' in locals():
    print("Found 'val_loader', using it as test_loader.")
    test_loader = val_loader
else:
    print("Creating a synthetic test_loader for speed measurement...")
    # Create 32 dummy images of size (1, 128, 128) just to measure speed
    # This ensures the code runs even if you lost your original X_test variables
    dummy_X = torch.randn(32, 1, 128, 128)
    dummy_y = torch.randint(0, 3, (32,))

    dummy_dataset = TensorDataset(dummy_X, dummy_y)
    test_loader = DataLoader(dummy_dataset, batch_size=32, shuffle=False)

Found 'val_loader', using it as test_loader.


In [25]:
import json
import pandas as pd
import numpy as np
import time
import torch

# =========================================================
# PART 1: Fulfill Prerequisite (Load Data from stats.json)
# =========================================================

# Load the stats file generated by your labeling code
try:
    with open("stats.json", "r") as f:
        stats_data = json.load(f)
    print(f"Successfully loaded stats for {len(stats_data)} matrices.")
except FileNotFoundError:
    print("Error: 'stats.json' not found. Please ensure the file from your labeling step is in the directory.")
    # Stop execution if file is missing (in a real notebook, you'd stop here)
    stats_data = {}

# Convert JSON data into a Pandas DataFrame
rows = []
for matrix_name, methods in stats_data.items():
    # Extract times for the 3 classes
    # Recall your structure: methods is a list of dicts with 'label' 0, 1, 2
    # We sort by label to ensure index 0 is Class 0, etc.
    sorted_methods = sorted(methods, key=lambda x: x['label'])

    t_0 = sorted_methods[0]['time']  # Entry-wise
    t_1 = sorted_methods[1]['time']  # Row-wise
    t_2 = sorted_methods[2]['time']  # Adaptive

    rows.append({
        'matrix_name': matrix_name,
        'time_0': t_0 * 1000,  # Convert seconds to milliseconds
        'time_1': t_1 * 1000,
        'time_2': t_2 * 1000
    })

df = pd.DataFrame(rows)

# =========================================================
# PART 2: Measure CNN Inference Time (The "Cost" of AI)
# =========================================================

def measure_cnn_overhead(model, loader, device):
    model.eval()

    # Warm-up (Crucial for accurate microsecond timing)
    dummy = torch.randn(1, 1, 128, 128).to(device)
    for _ in range(20):
        _ = model(dummy)

    print("Measuring CNN inference speed on GPU...")
    t0 = time.time()
    count = 0
    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            _ = model(inputs)
            count += inputs.size(0)

    torch.cuda.synchronize() # Wait for GPU to finish all tasks
    total_time = time.time() - t0

    # Average time per matrix in MICROseconds
    # Avoid division by zero if loader is empty
    if count == 0: return 0.0
    avg_us = (total_time / count) * 1e6
    return avg_us

# Run measurement (using your existing model and test_loader)
cnn_time_us = measure_cnn_overhead(model, test_loader, device)
print(f"Average CNN Inference Time: {cnn_time_us:.2f} microseconds")


# =========================================================
# PART 3: Comparative Analysis (The Answer for your Guide)
# =========================================================

print("\n" + "="*80)
print(f"{'Matrix Name':<30} | {'Ref. Time (Brute Force)':<25} | {'CNN Time (Predicted)':<20} | {'Speedup':<10}")
print("="*80)

# Pick 5 random matrices to show as "Case Studies"
if not df.empty:
    sample_indices = np.random.choice(df.index, 20, replace=False)
    samples = df.loc[sample_indices]

    for idx, row in samples.iterrows():
        # 1. Reference Time: The time it takes to run ALL 3 strategies on GPU
        # to empirically find the best one.
        ref_time_ms = row['time_0'] + row['time_1'] + row['time_2']

        # 2. CNN Time: Convert microseconds to milliseconds
        cnn_time_ms = cnn_time_us / 1000.0

        # 3. Speedup Factor
        speedup = ref_time_ms / cnn_time_ms if cnn_time_ms > 0 else 0

        # Truncate name for display
        disp_name = (row['matrix_name'][:27] + '..') if len(row['matrix_name']) > 29 else row['matrix_name']

        print(f"{disp_name:<30} | {ref_time_ms:.4f} ms{'':<18} | {cnn_time_ms:.4f} ms{'':<13} | {speedup:.0f}x")
else:
    print("DataFrame is empty. Please check stats.json.")

print("="*80)
print(f"\nAnalysis for Thesis/Defense:")
print(f"1. To discover the optimal strategy without AI, we must run the SpMV kernel 3 times.")
print(f"   On average, this 'Reference Discovery' takes {df[['time_0','time_1','time_2']].sum(axis=1).mean():.2f} ms per matrix.")
print(f"2. With PrecisionCNN, we predict the strategy in {cnn_time_us:.2f} microseconds.")
print(f"3. This results in a speedup of {df[['time_0','time_1','time_2']].sum(axis=1).mean() / (cnn_time_us/1000):.0f}x for the decision-making process.")

Successfully loaded stats for 1211 matrices.
Measuring CNN inference speed on GPU...
Average CNN Inference Time: 217.63 microseconds

Matrix Name                    | Ref. Time (Brute Force)   | CNN Time (Predicted) | Speedup   
HB/bcsstk04                    | 0.5744 ms                   | 0.2176 ms              | 3x
GHS_indef/aug3d                | 0.9713 ms                   | 0.2176 ms              | 4x
Cannizzo/sts4098               | 1.0679 ms                   | 0.2176 ms              | 5x
JGD_Kocay/Trec7                | 0.6118 ms                   | 0.2176 ms              | 3x
GHS_indef/ncvxqp1              | 0.5080 ms                   | 0.2176 ms              | 2x
HB/hor_131                     | 0.5056 ms                   | 0.2176 ms              | 2x
Hollinger/mark3jac140          | 1.1184 ms                   | 0.2176 ms              | 5x
HB/mbeaflw                     | 0.5575 ms                   | 0.2176 ms              | 3x
Hollinger/jan99jac080sc        | 0.5866 ms 

In [26]:
# Filter and Sort to find the "Hero" matrices
# 1. Calculate Reference Time for ALL matrices first
df['ref_time_ms'] = df['time_0'] + df['time_1'] + df['time_2']
df['cnn_time_ms'] = cnn_time_us / 1000.0  # Constant overhead
df['speedup'] = df['ref_time_ms'] / df['cnn_time_ms']

# 2. Get the Top 10 matrices with the SLOWEST execution time (Largest matrices)
hero_matrices = df.sort_values(by='ref_time_ms', ascending=False).head(10)

print("\n" + "="*105) # Increased length for new column
print(f"{'Matrix Name (Large)':<30} | {'Ref. Time (Brute Force)':<25} | {'CNN Time (Predicted)':<22} | {'Speedup':<10}")
print("="*105)

for idx, row in hero_matrices.iterrows():
    disp_name = (row['matrix_name'][:27] + '..') if len(row['matrix_name']) > 29 else row['matrix_name']

    # Print row with all three values
    print(f"{disp_name:<30} | {row['ref_time_ms']:.4f} ms{'':<18} | {row['cnn_time_ms']:.4f} ms{'':<15} | {row['speedup']:.0f}x")

print("="*105)

# 3. Calculate "Potential" Speedup for Large Scale Problems
avg_large_speedup = hero_matrices['speedup'].mean()
print(f"\nAnalysis for Thesis Defense:")
print(f"While the average speedup across the entire dataset is ~2x due to small matrix overhead,")
print(f"for computationally intensive (large) matrices, the speedup jumps to roughly {avg_large_speedup:.0f}x.")
print(f"This proves the CNN approach becomes exponentially more valuable as problem size increases.")


Matrix Name (Large)            | Ref. Time (Brute Force)   | CNN Time (Predicted)   | Speedup   
Mallya/lhr04                   | 3.3858 ms                   | 0.2176 ms                | 16x
Sandia/ASIC_100ks              | 3.2135 ms                   | 0.2176 ms                | 15x
Hollinger/mark3jac100sc        | 2.9401 ms                   | 0.2176 ms                | 14x
Nemeth/nemeth02                | 2.8216 ms                   | 0.2176 ms                | 13x
Nemeth/nemeth03                | 2.7242 ms                   | 0.2176 ms                | 13x
Hollinger/mark3jac120          | 2.6942 ms                   | 0.2176 ms                | 12x
Hollinger/mark3jac100          | 2.5332 ms                   | 0.2176 ms                | 12x
Hollinger/mark3jac080sc        | 2.4245 ms                   | 0.2176 ms                | 11x
LPnetlib/lp_dfl001             | 2.2851 ms                   | 0.2176 ms                | 11x
Oberwolfach/rail_79841         | 2.2817 ms              

In [27]:
# Filter and Sort to find the "Hero" matrices
# 1. Calculate Reference Time for ALL matrices first
df['ref_time_ms'] = df['time_0'] + df['time_1'] + df['time_2']
df['cnn_time_ms'] = cnn_time_us / 1000.0  # Constant overhead
df['speedup'] = df['ref_time_ms'] / df['cnn_time_ms']

# 2. Get the Top 10 matrices with the SLOWEST execution time (Largest matrices)
hero_matrices = df.sort_values(by='ref_time_ms', ascending=True).head(10)

print("\n" + "="*105) # Increased length for new column
print(f"{'Matrix Name (Large)':<30} | {'Ref. Time (Brute Force)':<25} | {'CNN Time (Predicted)':<22} | {'Speedup':<10}")
print("="*105)

for idx, row in hero_matrices.iterrows():
    disp_name = (row['matrix_name'][:27] + '..') if len(row['matrix_name']) > 29 else row['matrix_name']

    # Print row with all three values
    print(f"{disp_name:<30} | {row['ref_time_ms']:.4f} ms{'':<18} | {row['cnn_time_ms']:.4f} ms{'':<15} | {row['speedup']:.0f}x")

print("="*105)

# 3. Calculate "Potential" Speedup for Large Scale Problems
avg_large_speedup = hero_matrices['speedup'].mean()
print(f"\nAnalysis for Thesis Defense:")
print(f"While the average speedup across the entire dataset is ~2x due to small matrix overhead,")
print(f"for computationally intensive (large) matrices, the speedup jumps to roughly {avg_large_speedup:.0f}x.")
print(f"This proves the CNN approach becomes exponentially more valuable as problem size increases.")




Matrix Name (Large)            | Ref. Time (Brute Force)   | CNN Time (Predicted)   | Speedup   
JGD_Homology/n3c4-b3           | 0.3722 ms                   | 0.2176 ms                | 2x
JGD_Homology/n3c4-b2           | 0.3788 ms                   | 0.2176 ms                | 2x
JGD_Homology/n3c5-b7           | 0.3791 ms                   | 0.2176 ms                | 2x
JGD_Homology/n3c6-b11          | 0.3835 ms                   | 0.2176 ms                | 2x
Gset/G19                       | 0.3838 ms                   | 0.2176 ms                | 2x
Rommes/S20PI_n1                | 0.3862 ms                   | 0.2176 ms                | 2x
Gset/G41                       | 0.3919 ms                   | 0.2176 ms                | 2x
JGD_Homology/n3c5-b6           | 0.3941 ms                   | 0.2176 ms                | 2x
JGD_Homology/n3c4-b4           | 0.3947 ms                   | 0.2176 ms                | 2x
JGD_Homology/n3c5-b2           | 0.3952 ms                   | 0.